In [ ]:
import os
import mne
import mne_bids
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib.cm as cm

from glob import glob
from mne_bids import BIDSPath, read_raw_bids
from scipy.stats import ttest_rel

import matplotlib.patches as mpatches
%matplotlib widget


### define inputs

In [ ]:
bids_root = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1\data-bids"
print('BIDS root directory:', bids_root)
events_dir = r"C:\Users\Laura\Documents\PhD\Soundbrain lab\EAM\stimtrack-compare\python_v2"

# derivatives from status
deriv_dir_motor = os.path.join(bids_root, 'derivatives', 'cortical_filtered_motor')
if not os.path.exists(deriv_dir_motor):
    os.makedirs(deriv_dir_motor)

deriv_dir_status = os.path.join(bids_root, 'derivatives', 'cortical_filtered_status')
if not os.path.exists(deriv_dir_status):
    os.makedirs(deriv_dir_status) 

# derivatives from stimtrack
deriv_dir = os.path.join(bids_root, 'derivatives', 'cortical_filtered')
if not os.path.exists(deriv_dir):
    os.makedirs(deriv_dir)

In [ ]:
len(os.listdir(deriv_dir_status))

# Preprocess data (BIDS)

In [ ]:
task_list = ['active', 'passive', 'motor']
bl_start = 0.100
l_freq = 1
h_freq = 30
passive_hex = '#4477AA'
active_hex = '#CC6677'

### Cortically filtered (1–30 Hz)

In [ ]:

# for sub_num in range(2, 35):
for sub_num in [9]:
    sub_label = f'{sub_num:02d}'

    # initialize an empty dictionary for data
    task_evoked_dict = {}
    event_evoked_dict = {}

    for task_label in task_list:
        print(f'Loading {task_label} data')

        epoch_list = []

        for run_label in range(1,6):
        # load in EEG data
            bids_path = BIDSPath(root=bids_root, datatype='eeg', 
                                    subject=str(sub_label), 
                                    task=task_label, 
                                    run=run_label)
            
            try:
                data = read_raw_bids(bids_path)
                data.load_data()

                # re-reference data to linked mastoid reference
                data_ref = data.set_eeg_reference(ref_channels=['M1', 'M2'])
                
                # filter data
                data_filtered = data_ref.copy().filter(l_freq=l_freq, h_freq=h_freq)


                # loading events from the stim-track-generated events
                events_fpath = os.path.join(events_dir, f'sub-{sub_label}', f'sub-{sub_label}_task-{task_label}_run-{run_label}_events.tsv')
                events_df = pd.read_csv(events_fpath, delimiter='\t') 
                annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.type)
                data_filtered.set_annotations(annot)
                events_from_annot, event_dict2 = mne.events_from_annotations(data_filtered)
                
                # epoch data based on stimulus events
                epochs = mne.Epochs(data_filtered, 
                                    events=events_from_annot, #events, 
                                    event_id=event_dict2, #event_dict,
                                    on_missing='warn',
                                    picks=['Cz'],
                                    tmin=-1*bl_start, tmax=0.4, 
                                    baseline=[-1*bl_start, 0],
                                    reject=dict(eeg=75e-6)).drop_bad()
                
                epoch_list.append(epochs)

            except Exception as e:
                print(f"No run {run_label} for task-{task_label}: {e}")
        
        try:
            # combine epochs across runs
            all_epochs = mne.concatenate_epochs(epoch_list)
        except IndexError as e:
            print(f'cannot run for sub-{sub_label} task-{task_label} run-{run_label}: {e}')

        try:
            out_base = f'sub-{sub_label}_task-{task_label}_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif'
            all_epochs.save(os.path.join(deriv_dir, out_base), overwrite=True)
        except Exception as e:
            print(f"Error saving epochs for sub-{sub_label} task-{task_label}: {e}")
            continue


### Open-filter

In [ ]:

for sub_num in range(2, 35):
    sub_label = f'{sub_num:02d}'

    # initialize an empty dictionary for data
    task_evoked_dict = {}
    event_evoked_dict = {}

    for task_label in task_list:
        print(f'Loading {task_label} data')

        epoch_list = []

        for run_label in range(1,6):
        # load in EEG data
            bids_path = BIDSPath(root=bids_root, datatype='eeg', 
                                 subject=str(sub_label), 
                                 task=task_label, 
                                 run=run_label)
            
            try:
                data = read_raw_bids(bids_path)
                data.load_data()

            except:
                print(f"No run {run_label} for task-{task_label}")

            try:
                # re-reference data to linked mastoid reference
                data_ref = data.set_eeg_reference(ref_channels=['M1', 'M2'])
                
                # DO NOT filter data
                data_filtered = data_ref.copy()#.filter(l_freq=l_freq, h_freq=h_freq)


                # loading events from the stim-track-generated events
                events_fpath = sorted(glob(os.path.join(bids_root, 'derivatives', 'events-stimtrack') + 
                        f'/*{sub_label}*{task_label}*{run_label}_stimtrack_events.tsv'))[0]
                events_df = pd.read_csv(events_fpath, delimiter='\t') 
                annot = mne.Annotations(onset=events_df.onset, duration=0.170, description=events_df.type)
                data_filtered.set_annotations(annot)
                events_from_annot, event_dict2 = mne.events_from_annotations(data_filtered)
                
                # epoch data based on stimulus events
                epochs = mne.Epochs(data_filtered, 
                                    events=events_from_annot, #events, 
                                    event_id=event_dict2, #event_dict,
                                    on_missing='warn',
                                    picks=['Cz'],
                                    tmin=-1*bl_start, tmax=0.4, 
                                    baseline=[-1*bl_start, 0],
                                    reject=dict(eeg=75e-6)).drop_bad()
                
                epoch_list.append(epochs)
            except:
                print(f'Error occurred while processing sub-{sub_label} task-{task_label} run-{run_label}')

        try:
            # combine epochs across runs
            all_epochs = mne.concatenate_epochs(epoch_list)
            print(all_epochs)
        except:
            print(f'cannot combine epochs for sub-{sub_label} task-{task_label} run-{run_label}')

        try:
            out_base = f'sub-{sub_label}_task-{task_label}_baseline-{bl_start}_filt-open_run-all_event-stimtrack_epo.fif'
            all_epochs.save(os.path.join(deriv_dir, out_base), overwrite=True)
        except:
            continue


# Analyze whole-group data

Stimtrack epochs

In [ ]:
act_pos_avgs = []
act_neg_avgs = []
act_eps = [sorted(glob(deriv_dir+f'/sub-{sub_num:02d}_task-active_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif') for sub_num in range(2,35))]
for act_ep in act_eps[0]:
    try:
        sub_active_epochs = mne.read_epochs(act_ep[0], verbose='WARNING')
        act_pos_avgs.append(sub_active_epochs['1'].average())
        act_neg_avgs.append(sub_active_epochs['2'].average())
    except:
        pass

pas_pos_avgs = []
pas_neg_avgs = []
pas_eps = [sorted(glob(deriv_dir+f'/sub-{sub_num:02d}_task-passive_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-stimtrack_epo.fif') for sub_num in range(2,35))]
for pas_ep in pas_eps[0]:
    try:
        sub_passive_epochs = mne.read_epochs(pas_ep[0], verbose='WARNING')
        pas_pos_avgs.append(sub_passive_epochs['1'].average())
        pas_neg_avgs.append(sub_passive_epochs['2'].average())
    except:
        pass

Triggy Epochs (using Status channel)

In [ ]:
act_pos_avgs_st = []
act_neg_avgs_st = []
act_eps = [sorted(glob(deriv_dir_status+f'/sub-{sub_num:02d}_task-active_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif') for sub_num in range(2,35))]
for act_ep in act_eps[0]:
    try:
        sub_active_epochs = mne.read_epochs(act_ep[0], verbose='WARNING')
        act_neg_avgs_st.append(sub_active_epochs['active/neg'].average())
        act_pos_avgs_st.append(sub_active_epochs['active/pos'].average())
    except:
        pass

pas_pos_avgs_st = []
pas_neg_avgs_st = []
pas_eps = [sorted(glob(deriv_dir_status+f'/sub-{sub_num:02d}_task-passive_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif') for sub_num in range(2,35))]
for pas_ep in pas_eps[0]:
    try:
        sub_passive_epochs = mne.read_epochs(pas_ep[0], verbose='WARNING')
        pas_neg_avgs_st.append(sub_passive_epochs['passive/neg'].average())
        pas_pos_avgs_st.append(sub_passive_epochs['passive/pos'].average())
    except:
        pass

motor_avgs_st = []
motor_eps = sorted(glob(deriv_dir_motor+f'/sub-{sub_num:02d}_task-motor_baseline-{bl_start}_filt-{l_freq}-{h_freq}_run-all_event-Status_epo.fif') for sub_num in range(2, 35))
for motor_ep in motor_eps:
    try:
        sub_motor_epoch = mne.read_epochs(motor_ep[0], verbose=False)
        motor_avgs_st.append(sub_motor_epoch.average())
    except Exception as e:
        print(f"Error occurred: {e}")
        pass

In [ ]:
act_pos_avgs[:10]

In [ ]:
# average polarities within each subject
passive_avgs = [mne.combine_evoked([pas_pos_avgs[x], pas_neg_avgs[x]], weights='nave')
                for x in range(len(pas_pos_avgs))]
active_avgs = [mne.combine_evoked([act_pos_avgs[x], act_neg_avgs[x]], weights='nave')
                for x in range(len(act_pos_avgs))]

passive_avgs_st = [mne.combine_evoked([pas_pos_avgs_st[x], pas_neg_avgs_st[x]], weights='nave')
                for x in range(len(pas_pos_avgs_st))]
active_avgs_st = [mne.combine_evoked([act_pos_avgs_st[x], act_neg_avgs_st[x]], weights='nave')
                for x in range(len(act_pos_avgs_st))]

In [ ]:
picks = 'Cz'  # or a list of channels, or 'eeg'

conditions = {
    'Active': active_avgs ,
    'Passive': passive_avgs,
    'Motor': motor_avgs_st,
    'Active_status': active_avgs_st,
    'Passive_status': passive_avgs_st
}

In [ ]:
# plot each individual's evoked from stimtrack vs status channel - active
plt.close('all')
all_ids = list(range(2, 35))
skip_ids = [8, 10]
valid_ids = [s for s in all_ids if s not in skip_ids]

for sub_id, stim_avg, status_avg in zip(valid_ids, active_avgs, active_avgs_st):
    ch_idx = stim_avg.ch_names.index('Cz')

    plt.figure()
    plt.plot(stim_avg.times,
            stim_avg.get_data()[ch_idx] * 1e6, 
            alpha=0.4, 
            linewidth=0.8,
            label='stimtrack')
    plt.plot(status_avg.times,
             status_avg.get_data()[ch_idx] * 1e6, 
             alpha=0.4, 
             linewidth=0.8,
             label='triggy')
    plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
    plt.axvline(0, color='black', linewidth=0.5, linestyle='--')
    plt.title(f'{sub_id}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (µV)')
    plt.legend()
    plt.tight_layout()
    plt.show()
     

In [ ]:
# plot each individual's evoked from stimtrack vs status channel - passive
plt.close('all')
all_ids = list(range(2, 35))
skip_ids = [8, 10]
valid_ids = [s for s in all_ids if s not in skip_ids]

for sub_id, stim_avg, status_avg in zip(valid_ids, passive_avgs, passive_avgs_st):
    ch_idx = stim_avg.ch_names.index('Cz')

    plt.figure()
    plt.plot(stim_avg.times,
            stim_avg.get_data()[ch_idx] * 1e6, 
            alpha=0.4, 
            linewidth=0.8,
            label='stimtrack')
    plt.plot(status_avg.times,
             status_avg.get_data()[ch_idx] * 1e6, 
             alpha=0.4, 
             linewidth=0.8,
             label='triggy')
    plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
    plt.axvline(0, color='black', linewidth=0.5, linestyle='--')
    plt.title(f'{sub_id}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (µV)')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:

for cond_name, evokeds in conditions.items():
    # generate one color per subject from a colormap
    colors = cm.Blues(np.linspace(0.3, 0.8, len(evokeds)))
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ch_idx = evokeds[0].ch_names.index('Cz')
    
    for evoked, color in zip(evokeds, colors):
        ax.plot(evoked.times, evoked.get_data()[ch_idx] * 1e6,
                color=color, alpha=0.4, linewidth=0.8)
    
    grand_avg = mne.grand_average(evokeds)
    ax.plot(grand_avg.times, grand_avg.get_data()[ch_idx] * 1e6,
            color='navy', linewidth=2, label='Grand Average')
    
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set(xlabel='Time (s)', ylabel='Amplitude (µV)', title=cond_name)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# pas_pos_grandavg = mne.grand_average(pas_pos_avgs)
# pas_neg_grandavg = mne.grand_average(pas_neg_avgs)
# act_pos_grandavg = mne.grand_average(act_pos_avgs)
# act_neg_grandavg = mne.grand_average(act_neg_avgs)

# passive_grandavg = mne.grand_average(pas_pos_avgs+pas_neg_avgs)
# active_grandavg = mne.grand_average(act_pos_avgs+act_neg_avgs)

passive_grandavg = mne.grand_average(pas_pos_avgs_st+pas_neg_avgs_st)
active_grandavg = mne.grand_average(act_pos_avgs_st+act_neg_avgs_st)

motor_grandavg = mne.grand_average(motor_avgs_st)

## Combined polarities

In [ ]:
evokeds = dict(passive=pas_pos_avgs+pas_neg_avgs, active=act_pos_avgs+act_neg_avgs)

In [ ]:
mne.viz.plot_compare_evokeds(evokeds, 
                             truncate_xaxis=False, 
                             picks='Cz', 
                             ci=0.95)

### Motor subtraction

In [ ]:
print(active_grandavg.info['sfreq'] == motor_grandavg.info['sfreq'])
print(active_grandavg.times.shape == motor_grandavg.times.shape)
print(active_grandavg.ch_names == motor_grandavg.ch_names)

subtracted = mne.combine_evoked([active_grandavg, motor_grandavg], weights=[1, -1])
added = mne.combine_evoked([passive_grandavg, motor_grandavg], weights=[1, 1])

evokeds_dict = {
    'Active': active_grandavg,
    'Passive': passive_grandavg,
    'Motor': motor_grandavg,
    'Active - Motor': subtracted,
    'Passive + Motor': added
}

In [ ]:
# plot all (active, passive, motor, subtraction, addition)
mne.viz.plot_compare_evokeds(
    evokeds_dict,
    picks='Cz',
    combine='mean',
    legend=True
)

In [ ]:
# Power spectra
passive_psd = passive_grandavg.compute_psd(tmin=0.0, tmax=0.300, fmin=1, fmax=30)
active_psd = active_grandavg.compute_psd(tmin=0.0, tmax=0.300, fmin=1, fmax=30)

In [ ]:
# Spectrograms
frequencies = np.arange(4, 20)

passive_power = passive_grandavg.compute_tfr('multitaper', 
                                             freqs=frequencies, 
                                             n_cycles=1)
active_power = active_grandavg.compute_tfr('multitaper', 
                                           freqs=frequencies, 
                                           n_cycles=1)

In [ ]:
print(len(active_avgs_st), len(motor_avgs_st))
subtracted = []
for i in range(len(active_avgs_st)):
    subtracted.append(mne.combine_evoked([active_avgs_st[i], motor_avgs_st[i]], weights=[1, -1]))

### Peak stats

In [ ]:
f_low=4
f_high=15
t_low=0.00
t_high=0.16

# Extract power for passive condition
pas_peak_lat_list = []
pas_peak_amp_list = []
for evk in passive_avgs:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    pas_peak_lat_list.append(peak[1])
    pas_peak_amp_list.append(peak[2])

# Extract power for active condition
act_peak_lat_list = []
act_peak_amp_list = []
for evk in active_avgs:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    act_peak_lat_list.append(peak[1])
    act_peak_amp_list.append(peak[2])

# Create a DataFrame for power values
peak_lat_dict = {'passive': pas_peak_lat_list,  
              'active': act_peak_lat_list}
peak_df = pd.DataFrame(peak_lat_dict)

In [ ]:
passive_avgs[0].pick('Cz').get_peak()

In [ ]:
peak_df.head()

In [ ]:
# Compute the mean peak for each condition
peak_pass_mean = peak_df['passive'].mean()
peak_act_mean = peak_df['active'].mean()
print(f'Mean passive peak latency: {peak_pass_mean:.03f} s')
print(f'Mean active peak latency: {peak_act_mean:.03f} s')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_df['passive'], peak_df['active'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")

In [ ]:
# Plotting the power values
plt.figure(figsize=(5,4), dpi=200)
sns.swarmplot(peak_df, edgecolor='k', linewidth=1)
sns.boxplot(peak_df, fliersize=0)
plt.ylabel(f'Latency ({t_low}–{t_high} s)')
plt.title('P1 latency in passive vs active conditions')
plt.show()

### Power stats

In [ ]:
def compute_power(avg_evoked, f_low=90, f_high=110, t_low=0.1, t_high=0.2):
    # Compute the spectrogram (time-frequency representation)
    power = mne.time_frequency.tfr_multitaper(avg_evoked, 
                                            freqs=np.arange(f_low, f_high, 1), 
                                            n_cycles=2, 
                                            time_bandwidth=4.0,
                                            average=True, 
                                            return_itc=False)

    # Select the time and frequency bands
    time_mask = (power.times >= t_low) & (power.times <= t_high)
    freq_mask = (power.freqs >= f_low) & (power.freqs <= f_high)

    # Extract the power within the specified bands
    selected_power = power.data[:, freq_mask, :][:, :, time_mask]

    # Average power over the selected time and frequency bands
    average_power = selected_power.mean(axis=2).mean(axis=1)[0]
    return average_power

In [ ]:
f_low=4
f_high=30
t_low=0.00
t_high=0.3

# Extract power for passive condition
pas_peak_lat_list = []
pas_peak_amp_list = []
for evk in passive_avgs_st:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    pas_peak_lat_list.append(peak[1])
    pas_peak_amp_list.append(peak[2])

# Extract power for active condition
act_peak_lat_list = []
act_peak_amp_list = []
for evk in active_avgs_st:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    act_peak_lat_list.append(peak[1])
    act_peak_amp_list.append(peak[2])


# motor 
mot_peak_lat_list = []
mot_peak_amp_list = []
for evk in subtracted:
    peak = evk.get_peak(tmin=t_low, tmax=t_high, return_amplitude=True)
    mot_peak_lat_list.append(peak[1])
    mot_peak_amp_list.append(peak[2])

# Create a DataFrame for power values
peak_lat_dict = {'passive': pas_peak_lat_list,  
                 'active': act_peak_lat_list,
                 'motor': mot_peak_lat_list}

peak_amp_dict = {'passive': pas_peak_amp_list,  
                 'active': act_peak_amp_list,
                 'motor': mot_peak_amp_list}
peak_lat_df = pd.DataFrame(peak_lat_dict)
peak_amp_df = pd.DataFrame(peak_amp_dict)

In [ ]:
# Compute the mean peak for each condition
peak_pass_mean = peak_lat_df['passive'].mean()
peak_sub_mean = peak_lat_df['motor'].mean()
print(f'Mean passive peak latency: {peak_pass_mean:.03f} s')
print(f'Mean active - subtracted peak latency: {peak_sub_mean:.03f} s')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_lat_df['passive'], peak_lat_df['motor'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")

In [ ]:
peak_df = pd.DataFrame(peak_amp_dict)
print(peak_df.head())

# Compute the mean peak for each condition
peak_pass_mean = peak_df['passive'].mean()
peak_sub_mean = peak_df['motor'].mean()
print(f'Mean passive peak amplitude: {peak_pass_mean} ')
print(f'Mean active - motor peak amplitude: {peak_sub_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(peak_df['passive'], peak_df['motor'])
print(f"peak t-statistic: {t_stat:.03f}, peak p-value: {p_value:.05f}")

#### motor response

In [ ]:
f_low=4
f_high=30
t_low=-0.04
t_high=0.1

# Extract power for passive condition
pas_power_list = []
for evk in passive_avgs_st:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    pas_power_list.append(power)

# Extract power for active condition
act_power_list = []
for evk in active_avgs_st:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    act_power_list.append(power)

# Create a DataFrame for power values
power_dict = {'passive': pas_power_list,  
              'active': act_power_list}
power_df = pd.DataFrame(power_dict)

In [ ]:
# check the power dataframe
power_df.head()

In [ ]:
# Compute the mean power for each condition
power_pass_mean = power_df['passive'].mean()
power_act_mean = power_df['active'].mean()
print(f'Mean passive power: {power_pass_mean}')
print(f'Mean active power: {power_act_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(power_df['passive'], power_df['active'])
print(f"power t-statistic: {t_stat:.03f}, p-value: {p_value:.05f}")

In [ ]:
# Plotting the power values
plt.figure(figsize=(5,4), dpi=200)
sns.swarmplot(power_df, edgecolor='k', linewidth=1, log_scale=True)
sns.boxplot(power_df, fliersize=0)
plt.ylabel(f'Power ({f_low}–{f_high} Hz; {t_low}–{t_high} s)')
plt.title('Peri-stimulus cortical power')
plt.show()

#### Whole epoch

In [ ]:
f_low=4
f_high=30
t_low=-0.04
t_high=0.4

# Extract power for passive condition
pas_power_list = []
for evk in pas_pos_avgs+pas_neg_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    pas_power_list.append(power)

# Extract power for active condition
act_power_list = []
for evk in act_pos_avgs+act_neg_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    act_power_list.append(power)

# Create a DataFrame for power values
power_dict = {'passive': pas_power_list,  
              'active': act_power_list}
power_df = pd.DataFrame(power_dict)

In [ ]:
# check the power dataframe
power_df.head()

In [ ]:
# Compute the mean power for each condition
power_pass_mean = power_df['passive'].mean()
power_act_mean = power_df['active'].mean()
print(f'Mean passive power: {power_pass_mean}')
print(f'Mean active power: {power_act_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(power_df['passive'], power_df['active'])
print(f"power t-statistic: {t_stat:.03f}, p-value: {p_value:.03f}")

In [ ]:
# Plotting the power values
plt.figure(figsize=(5,4), dpi=200)
sns.swarmplot(power_df, edgecolor='k', linewidth=1, log_scale=True)
sns.boxplot(power_df, fliersize=0)
plt.ylabel(f'Power ({f_low}–{f_high} Hz; {t_low}–{t_high} s)')
plt.title('Power in passive vs. active conditions')
plt.show()

#### Compute grand average power

In [ ]:
# Extract power for each condition
f_low=4
f_high=15
t_low=0.1
t_high=0.2

passive_evoked_power = compute_power(passive_grandavg, 
                                     f_low=f_low, f_high=f_high, 
                                     t_low=t_low, t_high=t_high)

active_evoked_power = compute_power(active_grandavg, 
                                    f_low=f_low, f_high=f_high, 
                                    t_low=t_low, t_high=t_high)


#### Alpha band

In [ ]:
f_low=8
f_high=12
t_low=-0.04
t_high=0.4

# Extract power for passive condition
pas_power_list = []
for evk in passive_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    pas_power_list.append(power)

# Extract power for active condition
act_power_list = []
for evk in active_avgs:
    power = compute_power(evk, f_low=f_low, f_high=f_high, t_low=t_low, t_high=t_high)
    act_power_list.append(power)

# Create a DataFrame for power values
power_dict = {'passive': pas_power_list,  
              'active': act_power_list}
power_df = pd.DataFrame(power_dict)

In [ ]:
# check the power dataframe
power_df.head()

In [ ]:
# Compute the mean power for each condition
power_pass_mean = power_df['passive'].mean()
power_act_mean = power_df['active'].mean()
print(f'Mean passive power: {power_pass_mean}')
print(f'Mean active power: {power_act_mean}')

# Perform paired t-test
t_stat, p_value = ttest_rel(power_df['passive'], power_df['active'])
print(f"power t-statistic: {t_stat:.03f}, p-value: {p_value:.03f}")

In [ ]:
# Plotting the power values
plt.figure(figsize=(5,4), dpi=200)
sns.swarmplot(power_df, edgecolor='k', linewidth=1, log_scale=True)
sns.boxplot(power_df, fliersize=0)
plt.ylabel(f'Power ({f_low}–{f_high} Hz; {t_low}–{t_high} s)')
plt.title('Power in passive vs. active conditions')
plt.show()

In [ ]:
plt.close("all")
fig, axes = plt.subplot_mosaic(
    [["ERP", "ERP", "latency_box", "power_box"],
     ["p_spectrogram", "a_spectrogram", "diff_spectrogram", 'diff_spectrogram']],
     constrained_layout=True,
     width_ratios=[2, 2, 1, 1],
     dpi=150,
     figsize=(8, 6)
)

SMALL  = 11    # tick labels
MEDIUM = 12    # axis labels, colorbar label  
LARGE  = 13    # subplot titles

plt.rcParams.update({
    'font.size':        SMALL,
    'axes.titlesize':   LARGE,
    'axes.labelsize':   MEDIUM,
    'xtick.labelsize':  SMALL,
    'ytick.labelsize':  SMALL,
})

# set figure legend
fig.get_layout_engine().set(rect=[0, 0, 1, 0.92], hspace=0.0, wspace=0.0)
passive_patch = mpatches.Patch(color=passive_hex, label='Passive')
active_patch  = mpatches.Patch(color=active_hex,  label='Active')

fig.legend(
    handles=[passive_patch, active_patch],
    loc='upper center',
    ncols=2,
    frameon=False,
    fontsize=10,
)

# ERP waveform
mne.viz.plot_compare_evokeds(evokeds, 
                             truncate_xaxis=False, 
                             picks='Cz', 
                             ci=0.95,
                             axes=axes['ERP'],
                             colors={'passive': passive_hex, 'active': active_hex},
                             legend=False)
axes["ERP"].axvline(x=peak_pass_mean, color=passive_hex, linestyle='--', linewidth=1)
axes["ERP"].axvline(x=peak_act_mean, color=active_hex, linestyle='--', linewidth=1)
axes["ERP"].set_title("")
axes["ERP"].set_ylabel('Amplitude (µV)')
axes["ERP"].tick_params(direction="in")
# axes["ERP"].set_ylim(-2, 4)

# TODO: draw vertical dotted lines at P1 for each condition and annotate
# TODO: include star to mark stat significance

# box plot for power
sns.swarmplot(power_df,  
              linewidth=0.5, 
              log_scale=True,
              ax=axes["power_box"],
              size=2,
              palette=[passive_hex, active_hex],
              edgecolor='k')
sns.boxplot(power_df, 
            fliersize=0,
            ax=axes["power_box"],
            palette=[passive_hex, active_hex],
            linecolor='k')
axes["power_box"].set_ylabel(f'Power ({f_low}–{f_high} Hz; {t_low}–{t_high} s)')
axes["power_box"].set_yscale('log')
axes["power_box"].set_xlabel('')
axes["power_box"].set_xticklabels([])
axes["power_box"].tick_params(direction="in")
axes["power_box"].set_xticks([])
axes["power_box"].set_ylim(1e-10, 1e-7)

# box plot for P1 latency
t_low=0.00
t_high=0.16
sns.swarmplot(peak_df,  
              linewidth=0.5, 
              ax=axes["latency_box"],
              size=2,
              palette=[passive_hex, active_hex],
              edgecolor='k')
sns.boxplot(peak_df, 
            fliersize=0,
            ax=axes["latency_box"],
            palette=[passive_hex, active_hex],
            linecolor='k')

axes["latency_box"].set_ylabel(f'Latency ({t_low}–{t_high} s)')
axes["latency_box"].set_xlabel('')
axes["latency_box"].set_xticklabels([])
axes["latency_box"].tick_params(direction="in")
axes["latency_box"].set_xticks([])
axes["latency_box"].set_ylim(0.04, 0.2)


diff_power = active_power.copy()
diff_power.data = active_power.data - passive_power.data

passive_power.plot(axes=axes["p_spectrogram"], 
                   tmax=0.4,
                   colorbar=False, 
                   show=False,
                   vlim=[0, 9e-9],
                   cmap='magma')
active_power.plot(axes=axes["a_spectrogram"], 
                  tmax=0.4,
                  colorbar=False, 
                  show=False,
                  vlim=[0, 9e-9],
                  cmap='magma')

diff_power.plot(axes=axes["diff_spectrogram"],
                tmax=0.4,
                colorbar=False,
                cmap='RdBu_r',  
                show=False)

axes["p_spectrogram"].set_title('Passive')
axes["a_spectrogram"].set_title('Active')
axes["p_spectrogram"].sharex(axes["diff_spectrogram"])
axes["a_spectrogram"].sharex(axes["diff_spectrogram"])
axes["p_spectrogram"].set_ylabel('Frequency (Hz)')
axes["a_spectrogram"].set_ylabel('')
axes["p_spectrogram"].set_xlabel('Time (s)')
axes["a_spectrogram"].set_xlabel('Time (s)')
axes["p_spectrogram"].tick_params(direction="in")
axes["a_spectrogram"].tick_params(direction="in")
axes["diff_spectrogram"].set_title('Active - Passive')
axes["diff_spectrogram"].set_xlabel('Time (s)')
axes["diff_spectrogram"].set_ylabel('')
axes["diff_spectrogram"].tick_params(direction="in")

# add a single shared colorbar between the two spectrograms
im = axes["p_spectrogram"].images[0]  # grab the image from one of the axes
cbar = fig.colorbar(im, ax=[axes["p_spectrogram"], axes["a_spectrogram"]], 
                    location='bottom',   # or 'right'
                    shrink=0.6,
                    label='Power ($\\times 10^{-9}$)')

im_diff = axes["diff_spectrogram"].images[0]
cbar_diff = fig.colorbar(im_diff, 
                         ax=axes["diff_spectrogram"], 
                         location='bottom',  
                           label='Power difference ($\\times 10^{-9}$)')

# Disconnect all MNE's interactive callbacks
fig.canvas.mpl_disconnect_all() if hasattr(fig.canvas, 'mpl_disconnect_all') else None
for cid in list(fig.canvas.callbacks.callbacks.get('scroll_event', {}).keys()):
    fig.canvas.mpl_disconnect(cid)

fig.savefig('cortical.svg', dpi=600)
